# Inspect One Batch And Inference

Loads one batch, builds masks, runs one forward pass, and computes one reconstruction loss.

In [ ]:
from pathlib import Path
import torch
from torch.utils.data import DataLoader
from research.masked_event_model.v1.config import DataConfig, ModelConfig, MaskConfig, LossConfig
from research.masked_event_model.v1.data import EventChunkDataset, discover_chunk_files, target_horizons_from_columns
from research.masked_event_model.v1.masking import build_structured_masks
from research.masked_event_model.v1.model import MaskedEventAutoencoder
from research.masked_event_model.v1.losses import masked_autoencoder_loss
from research.masked_event_model.v1.schema import QUOTE_FEATURE_COLUMNS, TRADE_FEATURE_COLUMNS, CHUNK_SUMMARY_COLUMNS

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
data_config = DataConfig(cache_root=Path(r'D:/market-data/flatfiles/us_stocks_sip/derived/event_chunks_v2'), max_files=8)
files = discover_chunk_files(data_config, start_date=data_config.train_start_date, end_date=data_config.validation_end_date)
import polars as pl
horizon_count = len(target_horizons_from_columns(pl.scan_parquet(str(files[0].path)).collect_schema().names()))
dataset = EventChunkDataset(config=data_config, split='train', batch_size=2, seed=17)
batch = next(iter(DataLoader(dataset, batch_size=None, num_workers=0)))
print({k: (v.shape if hasattr(v, 'shape') else type(v)) for k, v in batch.items()})

In [ ]:
model = MaskedEventAutoencoder(
    quote_feature_count=len(QUOTE_FEATURE_COLUMNS), trade_feature_count=len(TRADE_FEATURE_COLUMNS), summary_feature_count=len(CHUNK_SUMMARY_COLUMNS),
    context_chunks=data_config.context_chunks, max_quote_events=data_config.max_quote_events, max_trade_events=data_config.max_trade_events,
    max_total_events=data_config.max_total_events, horizon_count=horizon_count, target_bit_count=data_config.target_bit_count,
    config=ModelConfig(d_model=128, n_heads=4, quote_event_layers=1, trade_event_layers=1, temporal_layers=1, decoder_layers=1, ffn_mult=2),
).to(device)
moved = {k: (v.to(device) if hasattr(v, 'to') else v) for k, v in batch.items()}
masks = build_structured_masks(quote_values=moved['quote_values'], trade_values=moved['trade_values'], chunk_summary=moved['chunk_summary'], event_kinds=moved['event_kinds'], config=MaskConfig())
out = model(moved['quote_values'], moved['trade_values'], moved['event_kinds'], moved['event_indices'], moved['chunk_summary'], masks)
loss, metrics = masked_autoencoder_loss(out, moved, masks, LossConfig())
print('loss', float(loss.detach().cpu()))
print(metrics)
print('forecast_logits', out.forecast_logits.shape, 'embedding', out.embeddings.shape)